# 04 — Exploration et comparaison de modèles

**Projet** : Segmentation client d'un e-commerçant (RFM étendu) avec scikit-learn
**Modèle configuré** : `kmeans` (KMeans (initialisation k-means++))
**Pourquoi ce choix** : KMeans est l'algorithme de référence pour une première segmentation : il est rapide, déterministe à graine fixée, et ses centroïdes se lisent directement comme des profils en écarts-types — ce qui rend chaque groupe nommable par le CRM. L'initialisation k-means++ répétée (`n_init`) réduit fortement le risque d'optimum local, principal défaut de la méthode. Ses hypothèses (groupes convexes, de taille comparable, dans un espace standardisé) sont exactement celles que ce projet documente et vérifie : la comparaison avec un mélange gaussien, au notebook 04, montre ce que l'on gagne (probabilités d'appartenance, ellipsoïdes) et ce que l'on perd (lisibilité, coût).

En apprentissage non supervisé, la question « pourquoi cet algorithme ? » se dédouble :

* **combien de groupes ?** — le nombre de segments est un choix, pas une découverte automatique ;
* **quelle méthode ?** — à k fixé, les algorithmes ne produisent pas la même géométrie (sphères de
  taille comparable pour KMeans, ellipsoïdes et probabilités d'appartenance pour un mélange
  gaussien).

Ce notebook répond aux deux **par des chiffres** : plancher aléatoire, critères internes, taille
des groupes, stabilité entre graines.

## Objectifs pédagogiques

1. Commencer par le **plancher** : une affectation aléatoire de même taille (silhouette ≈ 0).
1. Choisir le nombre de groupes par triangulation (silhouette, Davies-Bouldin, coude d'inertie, taille minimale).
1. Comparer les algorithmes de la stack **à k fixé** : la comparaison n'a de sens qu'à structure égale.
1. Mesurer la **stabilité** des affectations entre graines : une segmentation instable ne pilote pas de campagnes.

**Objectifs transverses du dépôt**

- Construire un pipeline non supervisé sans fuite : les colonnes de diagnostic sont exclues des features par configuration.
- Choisir le nombre de groupes par triangulation (silhouette, Davies-Bouldin, coude d'inertie, taille minimale).
- Comprendre l'effet de l'échelle et des queues de distribution sur une distance euclidienne (log, winsorising, standardisation).

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.18)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    # Le pré-traitement supprime la colonne de groupe (identifiant, non modélisable), or une
    # métrique de classement se calcule **par groupe** puis se moyenne. Elle doit donc voyager à
    # côté des matrices, exactement comme dans `TrainPipeline` et dans les fixtures de tests.
    # `None` pour toute tâche sans structure de groupe : le comportement des autres projets est
    # inchangé.
    group_column = getattr(config.data, "group_column", None)

    def groups_of(split: pd.DataFrame | None) -> Any:
        if split is None or not group_column or group_column not in split.columns:
            return None
        return split[group_column].to_numpy()

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "groups_train": groups_of(enriched["train"]),
        "groups_val": groups_of(enriched["val"]),
        "groups_test": groups_of(enriched["test"]),
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

## 1. Le plancher d'abord : une affectation aléatoire

In [ ]:
from src.evaluation.evaluator import Evaluator
from src.models import build_model
from src.training.losses_metrics import MetricCalculator, MetricInputs

BASE_MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
_ = BASE_MODEL.fit(PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[])

EVALUATOR = Evaluator.from_config(BASE_MODEL, CONFIG.model_dump(), NB_PATHS)
random_baseline = EVALUATOR.compare_to_baseline(PREPARED["X_val"], baseline="random_assignment")
single_baseline = EVALUATOR.compare_to_baseline(PREPARED["X_val"], baseline="single_cluster")

calculator = MetricCalculator(task=CONFIG.metrics.task, metrics=CONFIG.metrics.all_metrics)
model_metrics = calculator.evaluate(
    MetricInputs(
        y_true=None,
        y_pred=BASE_MODEL.predict(PREPARED["X_val"]),
        X=PREPARED["X_val"],
    )
)

secondary = list(CONFIG.metrics.secondary[:2])
comparison = pd.DataFrame(
    {
        "segmentation": [
            f"{CONFIG.model.algorithm} (configuré)",
            "affectation aléatoire (même k)",
            "groupe unique",
        ],
        CONFIG.metrics.primary: [
            model_metrics.get(CONFIG.metrics.primary, float("nan")),
            random_baseline.get(f"baseline_{CONFIG.metrics.primary}", float("nan")),
            single_baseline.get(f"baseline_{CONFIG.metrics.primary}", float("nan")),
        ],
    }
)
for name in secondary:
    comparison[name] = [
        model_metrics.get(name, float("nan")),
        random_baseline.get(f"baseline_{name}", float("nan")),
        single_baseline.get(f"baseline_{name}", float("nan")),
    ]
comparison.round(4)

**Ce qu'il faut retenir**

- Une affectation aléatoire de même taille a une silhouette proche de 0 : c'est le **plancher**. Toute segmentation publiée doit s'en détacher nettement.
- Le groupe unique est dégénéré (silhouette indéfinie) : il rappelle que « ne pas segmenter » n'est pas une option mesurable, c'est une absence de modèle.
- La métrique de décision du projet est `silhouette` (sens : maximize) ; le seuil de qualité déclaré est 0.18.

## 2. Combien de groupes ? Triangulation des critères

In [ ]:
k_table = EVALUATOR.k_selection(PREPARED["X_train"])
display(k_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.2))

axes[0].plot(k_table["k"], k_table["silhouette"], marker="o", color="#005f73", label="silhouette")
axes[0].set_xlabel("nombre de groupes (k)")
axes[0].set_ylabel("silhouette", color="#005f73")
twin = axes[0].twinx()
twin.plot(k_table["k"], k_table["davies_bouldin"], marker="s", color="#ae2012")
twin.set_ylabel("Davies-Bouldin (plus bas = mieux)", color="#ae2012")
twin.grid(False)
best_k = int(k_table.loc[k_table["silhouette"].idxmax(), "k"])
axes[0].axvline(best_k, color="#94d2bd", linestyle="--")
axes[0].set_title(f"Pic de silhouette : k = {best_k}")

axes[1].plot(k_table["k"], k_table["inertia"], marker="o", color="#bb3e03")
axes[1].set_xlabel("nombre de groupes (k)")
axes[1].set_ylabel("inertie intra-cluster")
axes[1].set_title("Coude d'inertie et taille du plus petit groupe")
twin2 = axes[1].twinx()
twin2.plot(
    k_table["k"], k_table["min_cluster_share"] * 100, marker="^", color="#6a4c93", linewidth=1.2
)
twin2.axhline(EVALUATOR.min_cluster_share * 100, color="#6a4c93", linestyle=":")
twin2.set_ylabel("plus petit groupe (%)", color="#6a4c93")
twin2.grid(False)

fig.tight_layout()
plt.show()

configured_k = int((CONFIG.model.params or {}).get("n_clusters", best_k))
print(f"k retenu par la configuration : {configured_k}")
print(f"k du pic de silhouette        : {best_k}")
lowest_db_k = int(k_table.loc[k_table["davies_bouldin"].idxmin(), "k"])
print(f"Davies-Bouldin minimal        : k = {lowest_db_k}")
K_CHOSEN = configured_k

**Ce qu'il faut retenir**

- Les critères **ne convergent pas toujours** : pic de silhouette, minimum de Davies-Bouldin et coude d'inertie peuvent désigner des k différents. C'est normal — ils ne mesurent pas la même chose.
- La taille du plus petit groupe est un critère **métier** : sous ~3 % de la base, un segment ne justifie pas une campagne dédiée (coût, lisibilité, volume statistique).
- Le k retenu ici est celui de la configuration : l'écart avec le pic statistique est un arbitrage assumé et documenté, pas une erreur.

## 3. Comparaison des algorithmes de la stack (à k fixé)

In [ ]:
from src.models.factory import available_algorithms

ALGORITHMS = available_algorithms(CONFIG.metrics.task)
print(f"{len(ALGORITHMS)} algorithmes disponibles pour la tâche '{CONFIG.metrics.task}' :")
print(ALGORITHMS)

# Comparaison **loyale** : tous les algorithmes sont entraînés avec le MÊME nombre de groupes.
# `n_clusters` (KMeans) et `n_components` (mélange gaussien) sont fournis ensemble : chaque
# estimateur ignore le paramètre qui ne le concerne pas (journalisé en debug par la fabrique).
rows = []
for algorithm in ALGORITHMS:
    try:
        candidate = build_model(
            CONFIG,
            feature_names=PREPARED["feature_names"],
            algorithm=algorithm,
            params={"n_clusters": K_CHOSEN, "n_components": K_CHOSEN},
        )
        result = candidate.fit(
            PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[]
        )
        labels = candidate.predict(PREPARED["X_val"])
        values = calculator.evaluate(MetricInputs(y_true=None, y_pred=labels, X=PREPARED["X_val"]))
        shares = pd.Series(labels).value_counts(normalize=True)
        row = {
            "algorithme": algorithm,
            CONFIG.metrics.primary: values.get(CONFIG.metrics.primary, float("nan")),
        }
        for name in CONFIG.metrics.secondary[:2]:
            row[name] = values.get(name, float("nan"))
        row["plus petit groupe (%)"] = (
            round(float(shares.min()) * 100, 2) if len(shares) else float("nan")
        )
        row["groupes"] = int(pd.Series(labels).nunique())
        row["secondes"] = round(result.duration_seconds, 2)
        rows.append(row)
    except Exception as error:  # un algorithme incompatible ne doit pas casser l'exploration
        rows.append(
            {
                "algorithme": algorithm,
                CONFIG.metrics.primary: float("nan"),
                "erreur": str(error)[:90],
            }
        )

ascending = CONFIG.metrics.direction == "minimize"
ranking = pd.DataFrame(rows).sort_values(
    CONFIG.metrics.primary, ascending=ascending, na_position="last"
)
ranking.round(4)

**Ce qu'il faut retenir**

- KMeans suppose des groupes **convexes** de taille comparable ; un mélange gaussien autorise des ellipsoïdes et produit des probabilités d'appartenance (donc une confiance par client).
- MiniBatchKMeans échange un peu de qualité contre un coût mémoire constant : c'est l'option des bases de plusieurs millions de clients.
- Un écart de silhouette inférieur à 0.02 entre deux algorithmes n'est pas un signal : il faut le comparer à la variabilité entre graines (section 4).

## 4. Stabilité : les clients gardent-ils leur segment d'une graine à l'autre ?

In [ ]:
from sklearn.metrics import adjusted_rand_score

reference_labels = np.asarray(BASE_MODEL.predict(PREPARED["X_val"])).ravel()

stability_rows = []
for seed in (7, 21, 42):
    candidate = build_model(CONFIG, feature_names=PREPARED["feature_names"])
    candidate.random_state = seed
    _ = candidate.fit(PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[])
    labels = np.asarray(candidate.predict(PREPARED["X_val"])).ravel()
    values = calculator.evaluate(MetricInputs(y_true=None, y_pred=labels, X=PREPARED["X_val"]))
    stability_rows.append(
        {
            "graine": seed,
            "ari_vs_référence": round(float(adjusted_rand_score(reference_labels, labels)), 4),
            CONFIG.metrics.primary: round(
                float(values.get(CONFIG.metrics.primary, float("nan"))), 4
            ),
        }
    )

stability = pd.DataFrame(stability_rows)
display(stability)
mean_ari = float(stability["ari_vs_référence"].mean())
scores = stability[CONFIG.metrics.primary].to_numpy(dtype="float64")
print(f"ARI moyen entre graines : {mean_ari:.3f}")
print(
    f"{CONFIG.metrics.primary} : moyenne {np.nanmean(scores):.4f} ± {np.nanstd(scores, ddof=1):.4f}"
)

**Ce qu'il faut retenir**

- L'**ARI entre graines** mesure ce que le métier redoute le plus : un client qui change de segment d'un jour à l'autre reçoit des messages contradictoires.
- KMeans converge vers des optima locaux : augmenter `n_init` (nombre de ré-initialisations) est le premier levier de stabilité, avant tout changement d'algorithme.
- Règle pratique : ne pas célébrer un gain de silhouette inférieur à 2× l'écart-type observé entre graines.

## 5. Sensibilité aux hyperparamètres

In [ ]:
import itertools

# Grille déclarée dans le manifeste (injectée dans `conf/model/default.yaml`) : aucune valeur
# n'est codée en dur dans le notebook.
GRID = {"n_clusters": [4, 5, 6, 7], "n_init": [5, 20]}
combinations = list(itertools.product(*[GRID[name] for name in GRID]))
print(f"{len(combinations)} combinaisons testées sur {list(GRID)}")

grid_rows = []
for combination in combinations:
    params = dict(zip(GRID, combination, strict=True))
    candidate = build_model(CONFIG, feature_names=PREPARED["feature_names"], params=params)
    result = candidate.fit(
        PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[]
    )
    labels = candidate.predict(PREPARED["X_val"])
    values = calculator.evaluate(MetricInputs(y_true=None, y_pred=labels, X=PREPARED["X_val"]))
    shares = pd.Series(labels).value_counts(normalize=True)
    row = {str(name): str(value) for name, value in params.items()}
    row[CONFIG.metrics.primary] = values.get(CONFIG.metrics.primary, float("nan"))
    for name in CONFIG.metrics.secondary[:2]:
        row[name] = values.get(name, float("nan"))
    row["plus petit groupe (%)"] = (
        round(float(shares.min()) * 100, 2) if len(shares) else float("nan")
    )
    row["secondes"] = round(result.duration_seconds, 2)
    grid_rows.append(row)

grid_results = pd.DataFrame(grid_rows).sort_values(
    CONFIG.metrics.primary, ascending=ascending, na_position="last"
)
grid_results.round(4)

**Ce qu'il faut retenir**

- Le tri respecte le **sens** de la métrique (`direction: maximize` pour une silhouette) : un tri ascendant par défaut classerait les pires modèles en premier.
- Une grille se lit aussi par sa **dispersion** : si toutes les combinaisons se tiennent en 0.01 de silhouette, le modèle est robuste — et le réglage fin n'est pas le levier principal.
- Attention au sur-ajustement sur le critère interne : maximiser la silhouette peut produire des groupes géométriquement nets mais métier-inutiles (micro-groupes, séparation par la région).

## 6. Choix argumenté

| Critère | Lecture | Décision |
| --- | --- | --- |
| Silhouette (validation) | structure réelle vs bruit | doit dépasser le seuil du cas d'usage |
| Davies-Bouldin | compacité / séparation | plus bas = mieux, mais sensible aux outliers |
| Taille du plus petit groupe | exploitabilité CRM | ≥ 3 % de la base, sinon fusion |
| ARI entre graines | reproductibilité des affectations | ≥ 0.85 avant mise en production |
| Coût d'entraînement | fenêtre de batch nocturne | MiniBatchKMeans si > 1 M de clients |

**Règle de décision retenue** : choisir le plus petit k qui satisfait la silhouette et la
stabilité, puis vérifier la lisibilité métier des profils (notebook 06). Un k supérieur n'est
justifié que s'il sépare un enjeu d'action distinct.